
# ED Ops Pipeline (Kaggle/Papermill) — Sim → Train (Focal) → Cal → Threshold (PR/OPS) → Replay (Top‑K) → KPIs

**Kaggle-ready**: kernelspec metadata, non-interactive plotting, environment-aware paths, no widgets/inputs.  
Artifacts go to **`/kaggle/working/artifacts/`** (or `./artifacts` if not on Kaggle).


In [ ]:

# ---- Environment & reproducibility (Papermill-safe) ----
import os, sys, json, math, random, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from collections import Counter

import torch, torch.nn as nn, torch.nn.functional as F

# Repro
SEED = 4242
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Paths
ON_KAGGLE = Path('/kaggle/working').exists()
WORKDIR   = Path('/kaggle/working') if ON_KAGGLE else Path('.')
ARTS      = WORKDIR/'artifacts'; ARTS.mkdir(parents=True, exist_ok=True)

# Data dirs: prefer Kaggle input, then /mnt/data, else cwd
DATA_DIRS = [Path('/kaggle/input'), Path('/mnt/data'), WORKDIR]
def find_data_file(name):
    for d in DATA_DIRS:
        p = d/name
        if p.exists():
            return p
    return None

print({"on_kaggle": ON_KAGGLE, "workdir": str(WORKDIR), "arts": str(ARTS)})
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.metrics._ranking")  # PR curve zero-positive warning


In [ ]:

# ---- Config knobs ----
ACTIONS = ['NO_OP','ORDER_ECG','PERFORM_FAST','ORDER_LABS','ORDER_XR','ORDER_CT','REQUEST_CONSULT','REQUEST_BED']
FEATURE_NAMES = ['minute_of_day','cap_stale','ems','consult_delay_min',
                 'syn_chest_pain','syn_polytrauma','syn_neuro_deficit','syn_other',
                 'ecg_hint','fast_hint','ct_hint','ems_prealert','risk_score']

# Arrival profile (per hour weight)
LAMBDA_BY_HOUR = {0:1.0,1:1.0,2:0.9,3:0.9,4:0.9,5:1.1,6:1.6,7:2.2,8:3.3,9:3.6,10:3.3,11:3.1,
                  12:3.0,13:3.0,14:3.0,15:3.2,16:3.5,17:3.7,18:3.0,19:2.5,20:2.0,21:1.8,22:1.4,23:1.2}
_LSUM = sum(LAMBDA_BY_HOUR.values())

# Presets
PRESET = "ops_realistic"            # or "train_heavy"
TARGETS = {"ops_realistic": 100, "train_heavy": 300}
TARGET_DAILY_ARRIVALS = TARGETS.get(PRESET, 100)
LAMBDA_SCALE = TARGET_DAILY_ARRIVALS / _LSUM

# Window: use 3 days to stabilize cal/val for Papermill exports
N_DAYS = 3
SIM_MINUTES = N_DAYS * 24 * 60

# Ops constraints
RECALL_FLOOR = {'ORDER_ECG':0.85,'ORDER_CT':0.85,'PERFORM_FAST':0.85,'ORDER_LABS':0.60,'ORDER_XR':0.65}
FP_BUDGET_PER_H = {"REQUEST_CONSULT": 0.10, "REQUEST_BED": 0.05}

# Top-K + mutual exclusions
TOP_K = 2
MUTEX = {frozenset({'ORDER_CT','PERFORM_FAST'})}

# Focal loss
USE_FOCAL = True
GAMMA = 2.0

print(f"Preset={PRESET} | target/day={TARGET_DAILY_ARRIVALS} | scale={LAMBDA_SCALE:.3f} | sim_minutes={SIM_MINUTES}")


In [ ]:

# ---- Simulation or CSV loader (Papermill-safe) ----
PREV = {'syn_chest_pain':0.22,'syn_polytrauma':0.06,'syn_neuro_deficit':0.08,'syn_other':0.64}

def nonhom_poisson_arrivals(total_minutes, lam_by_hour, scale=1.0):
    out=[]
    for m in range(total_minutes):
        h=(m//60)%24
        lam = lam_by_hour.get(h,1.0)*scale/60.0
        if np.random.rand()<lam: out.append(m)
    return out

def one_hot_syndrome():
    r=np.random.rand()
    if r<PREV['syn_chest_pain']: return 1,0,0,0
    r-=PREV['syn_chest_pain']
    if r<PREV['syn_polytrauma']: return 0,1,0,0
    r-=PREV['syn_polytrauma']
    if r<PREV['syn_neuro_deficit']: return 0,0,1,0
    return 0,0,0,1

def gen_dataset(total_minutes=SIM_MINUTES, scale=LAMBDA_SCALE, seq_len=5):
    arr=nonhom_poisson_arrivals(total_minutes, LAMBDA_BY_HOUR, scale)
    X=[]; y=[]; ts=[]
    for t in arr:
        ems = int(np.random.rand()<0.55)
        ems_prealert = int(ems and (np.random.rand()<0.35))
        cap_stale = int(np.random.rand()<0.25)
        s_cp, s_poly, s_neuro, s_other = one_hot_syndrome()
        ecg_hint = int(s_cp or (np.random.rand()<0.15))
        fast_hint = int(s_poly or (np.random.rand()<0.10))
        ct_hint = int((s_poly or s_neuro) or (np.random.rand()<0.10))
        base_delay = np.random.normal(20, 8)  # minutes
        consult_delay_min_raw = max(0.0, base_delay - 6*ems_prealert + 5*cap_stale)
        risk_raw = np.clip(0.15 + 0.35*s_poly + 0.35*s_neuro + 0.10*ems + 0.10*ems_prealert, 0, 1)
        minute_of_day_raw = t % 1440

        minute_of_day = minute_of_day_raw/1440.0
        consult_delay_min = np.log1p(consult_delay_min_raw/10.0)
        risk = risk_raw

        feats = [minute_of_day, cap_stale, ems, consult_delay_min, s_cp, s_poly, s_neuro,
                 s_other, ecg_hint, fast_hint, ct_hint, ems_prealert, risk]

        # Base label
        if (ct_hint==1) and (s_neuro==1 or s_poly==1):
            base = np.random.choice(['ORDER_CT','PERFORM_FAST'], p=[0.85,0.15])
        elif (s_cp==1 and ecg_hint==1):
            base = 'ORDER_ECG'
        elif (risk>0.55 and ems==1):
            base = 'ORDER_LABS'
        elif (s_other==1 and fast_hint==0 and np.random.rand()<0.25):
            base = 'ORDER_XR'
        else:
            base = 'NO_OP'

        # Escalation to paging (slightly higher to ensure some pages in VAL)
        escalate = (base in ['ORDER_CT','PERFORM_FAST','ORDER_LABS']) and (risk>0.65) and (np.random.rand()<0.35)
        if escalate:
            base = np.random.choice(['REQUEST_CONSULT','REQUEST_BED'], p=[0.75,0.25])

        X.append(feats); y.append(ACTIONS.index(base)); ts.append(t)

    X=np.array(X, dtype=np.float32); y=np.array(y, dtype=np.int64); ts=np.array(ts, dtype=np.int64)
    X_seq = np.stack([X + 0.01*np.random.randn(*X.shape) for _ in range(seq_len)], axis=1).astype(np.float32)
    return X_seq, y, ts

# If CSVs exist, use them (Kaggle input); else simulate
csv_train = find_data_file('train_DE_full.csv')
csv_val   = find_data_file('val_DE_full.csv')
csv_test  = find_data_file('test_DE_full.csv')

if csv_train and csv_val and csv_test:
    print("Loading provided CSVs from:", csv_train.parent)
    def load_split(p):
        df = pd.read_csv(p)
        X = df[FEATURE_NAMES].values.astype('float32')
        y = df['label'].map({k:i for i,k in enumerate(ACTIONS)}).values.astype('int64')
        ts = df.get('ts', pd.Series(np.arange(len(df)))).values.astype('int64')
        X_seq = np.stack([X + 0.01*np.random.randn(*X.shape) for _ in range(5)], axis=1).astype('float32')
        return X_seq, y, ts
    Xtr_all, ytr_all, ttr_all = load_split(csv_train)
    Xcal_all, ycal_all, tcal_all = load_split(csv_val)   # treat supplied "val" as CAL
    Xv_all,  yv_all,  tv_all  = load_split(csv_test)     # treat supplied "test" as VAL
else:
    print("Simulating dataset...")
    X_all, y_all, ts_all = gen_dataset()
    print("Realized arrivals:", len(X_all), f"(~{len(X_all)/N_DAYS:.1f}/day)")
    idx = np.arange(len(y_all)); np.random.shuffle(idx)
    n_cal = max(1, int(0.20*len(idx)))
    n_val = max(1, int(0.20*len(idx)))
    cal_idx = idx[:n_cal]; val_idx = idx[n_cal:n_cal+n_val]; train_idx = idx[n_cal+n_val:]
    Xtr_all, ytr_all, ttr_all = X_all[train_idx], y_all[train_idx], ts_all[train_idx]
    Xcal_all, ycal_all, tcal_all = X_all[cal_idx], y_all[cal_idx], ts_all[cal_idx]
    Xv_all,  yv_all,  tv_all  = X_all[val_idx],  y_all[val_idx],  ts_all[val_idx]

print("Shapes  train/cal/val:", Xtr_all.shape, Xcal_all.shape, Xv_all.shape)
print("Class dist (train):", Counter(ytr_all))
print("Class dist (cal):  ", Counter(ycal_all))
print("Class dist (val):  ", Counter(yv_all))


In [ ]:

# ---- Model + Focal training (reports train & cal loss only) ----
device = torch.device("cpu")

class GRUHead(nn.Module):
    def __init__(self, in_f, hidden=128, n_actions=8, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(in_f, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_actions)
    def forward(self, x):
        out,_=self.gru(x); h=self.drop(out[:,-1,:]); return self.fc(h)

def class_weights(y, n_classes):
    cnt = np.bincount(y, minlength=n_classes).astype(float)
    inv = 1.0/np.sqrt(cnt + 1e-6)
    w = inv / inv.sum() * n_classes
    return torch.tensor(w, dtype=torch.float32)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction='none')
    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        with torch.no_grad():
            pt = torch.softmax(logits, dim=-1).gather(1, targets[:,None]).squeeze(1).clamp_min(1e-6)
        return ((1-pt)**self.gamma * ce).mean()

def to_t(x): return torch.tensor(x, dtype=torch.float32, device=device)
def to_y(x): return torch.tensor(x, dtype=torch.long, device=device)

model = GRUHead(len(FEATURE_NAMES), hidden=128, n_actions=len(ACTIONS), dropout=0.2).to(device)
weights = class_weights(ytr_all, len(ACTIONS)).to(device)
criterion = FocalLoss(gamma=GAMMA, weight=weights) if USE_FOCAL else nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.AdamW(model.parameters(), lr=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=35)

BATCH=256; EPOCHS=35
def batches(X,y,bs=BATCH):
    N=len(y); idx=np.arange(N); np.random.shuffle(idx)
    for i in range(0,N,bs):
        b=idx[i:i+bs]; yield X[b], y[b]

for ep in range(1,EPOCHS+1):
    model.train(); tl=0.0; tn=0
    for xb,yb in batches(Xtr_all, ytr_all, BATCH):
        xb_t, yb_t = to_t(xb), to_y(yb)
        opt.zero_grad(); loss = criterion(model(xb_t), yb_t); loss.backward(); opt.step()
        tl += float(loss.detach())*len(yb); tn += len(yb)
    model.eval()
    with torch.no_grad():
        vl = float(criterion(model(to_t(Xcal_all)), to_y(ycal_all)).detach())
    sched.step()
    if ep%5==0 or ep in [1,2,3]: print(f"Epoch {ep:02d}: train={tl/tn:.4f} cal={vl:.4f}")


In [ ]:

# ---- Calibration on CAL + thresholds (PR-cost & OPS) ----
@torch.no_grad()
def logits_on(splitX): return model(to_t(splitX)).cpu().numpy()

def softmaxT(z, T=1.0, axis=1):
    zT = z / max(T,1e-6)
    zT -= zT.max(axis=axis, keepdims=True)
    e = np.exp(zT)
    return e / e.sum(axis=axis, keepdims=True)

def nll(probs, y_true):
    idx=np.arange(len(y_true)); p=probs[idx, y_true]; return -np.log(np.clip(p,1e-9,1)).mean()

cal_logits = logits_on(Xcal_all)
Ts = np.linspace(0.6,1.8,25)
bestT, best = 1.0, 1e9
for T in Ts:
    p = softmaxT(cal_logits, T=T)
    cur = nll(p, ycal_all)
    if cur < best:
        best, bestT = cur, T
print(f"[Calib] Best T on CAL = {bestT:.3f}")

probs_cal = softmaxT(cal_logits, T=bestT)

def pick_tau_cost(y_true_bin, y_prob, fp_cost=1.0, fn_cost=5.0):
    taus = np.linspace(0,1,101); best=None
    for t in taus:
        yp=(y_prob>=t).astype(int)
        tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())
        prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
        cost=fp_cost*fp + fn_cost*fn
        if best is None or cost<best[0]: best=(cost,t,prec,rec,f1,tp,fp,fn)
    c,t,prec,rec,f1,tp,fp,fn=best
    return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
            "cost":float(c),"tp":tp,"fp":fp,"fn":fn}

def pick_tau_ops(y_true_bin, y_prob, ts_min, name):
    taus=np.linspace(0,1,101)
    hours=(ts_min.max()-ts_min.min()+1)/60.0
    best=None
    recall_floor = RECALL_FLOOR.get(name, None)
    fp_budget_per_h = FP_BUDGET_PER_H.get(name, None)
    for t in taus:
        yp=(y_prob>=t).astype(int)
        tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())
        prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
        fp_per_h = fp / max(hours,1e-6)
        if recall_floor is not None and rec < recall_floor: continue
        if fp_budget_per_h is not None and fp_per_h > fp_budget_per_h: continue
        score=f1
        if best is None or score>best[0]: best=(score,t,prec,rec,f1,tp,fp,fn,fp_per_h)
    if best is None:
        paging = name in ['REQUEST_CONSULT','REQUEST_BED']
        if not paging:
            # Relax recall floor for non-paging actions: choose best F1 without constraints
            best2=None
            for t in np.linspace(0,1,101):
                yp=(y_prob>=t).astype(int)
                tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())
                prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
                fp_per_h = fp / max(hours,1e-6)
                score=f1
                if best2 is None or score>best2[0]: best2=(score,t,prec,rec,f1,tp,fp,fn,fp_per_h)
            score,t,prec,rec,f1,tp,fp,fn,fp_per_h = best2
            return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
                    "tp":tp,"fp":fp,"fn":fn,"fp_per_h":float(fp_per_h),"status":"relaxed","relaxed_from":recall_floor}
        else:
            # Keep strict for paging actions; mark fallback
            t=0.5; yp=(y_prob>=t).astype(int)
            tp=int(((yp==1)&(y_true_bin==1)).sum()); fp=int(((yp==1)&(y_true_bin==0)).sum()); fn=int(((yp==0)&(y_true_bin==1)).sum())
            prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
            fp_per_h = fp / max(hours,1e-6)
            return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
                    "tp":tp,"fp":fp,"fn":fn,"fp_per_h":float(fp_per_h),"status":"fallback"}
    score,t,prec,rec,f1,tp,fp,fn,fp_per_h = best
    return {"tau":float(t),"precision":float(prec),"recall":float(rec),"f1":float(f1),
            "tp":tp,"fp":fp,"fn":fn,"fp_per_h":float(fp_per_h)}

thr_cost, thr_ops, notes = {}, {}, {}
for ci,name in enumerate(ACTIONS):
    yb=(ycal_all==ci).astype(int)
    thr_cost[name] = pick_tau_cost(yb, probs_cal[:,ci])
    thr_ops[name]  = pick_tau_ops(yb, probs_cal[:,ci], tcal_all, name)
    notes[name] = {'recall_floor': RECALL_FLOOR.get(name),
                   'fp_budget_per_h': FP_BUDGET_PER_H.get(name),
                   'achieved_ops': thr_ops[name]}

(ARTS/'thresholds.json').write_text(json.dumps(thr_cost, indent=2))
(ARTS/'thresholds_ops.json').write_text(json.dumps(thr_ops, indent=2))
(ARTS/'thresholds_ops_notes.json').write_text(json.dumps(notes, indent=2))

bad = [n for n,info in thr_ops.items() if info.get('status')=='fallback']
if bad:
    print(f"[WARN] Ops τ fallback for (paging strict): {bad}. Consider increasing N_DAYS or relaxing FP budgets.")
print("Saved thresholds & notes.")


In [ ]:

# ---- Replay on VAL (Top-K, capacity freshness, minutes saved) ----
def capacity_fresh(ts_now, last_update_ts, max_age_min=20):
    if last_update_ts is None: return False
    return (ts_now - last_update_ts) <= max_age_min

def simulate_capacity_updates(total_minutes=SIM_MINUTES):
    t=0; updates=[]
    while t<total_minutes:
        updates.append(t)
        t += int(np.random.uniform(10,20))
    return set(updates)

def minutes_saved_for(action, ts_now, last_cap_update):
    hour = (ts_now//60)%24
    is_day = 8 <= hour < 20
    if action in ['ORDER_CT','PERFORM_FAST']:
        base = 15.0 if is_day else 30.0
    elif action in ['REQUEST_CONSULT','REQUEST_BED']:
        base = 4.0 if is_day else 6.0
    elif action in ['ORDER_ECG','ORDER_LABS']:
        base = 2.0 if is_day else 3.0
    elif action in ['ORDER_XR']:
        base = 1.5 if is_day else 2.5
    else:
        base = 0.0
    if last_cap_update is None:
        return 0.0
    delta = ts_now - last_cap_update
    if delta <= 0: discount = 1.0
    elif delta >= 20: discount = 0.0
    else: discount = 1.0 - (delta/20.0)
    return max(0.0, base * discount)

@torch.no_grad()
def predict_probs_row(x_row, T):
    logits = model(to_t(x_row[None,...]))
    return torch.softmax(logits/T, dim=-1).cpu().numpy()[0]

def replay(thresholds, Tcal, X_ref, y_ref, ts_ref):
    logs={'proposals':[], 'gate_blocks':[], 'policy_blocks':[], 'approvals':[]}
    state={'last_capacity_update_ts':None, 'sent_packets':set()}
    caps=simulate_capacity_updates(SIM_MINUTES)
    order=np.argsort(ts_ref)
    for i in order:
        tnow=int(ts_ref[i])
        if tnow in caps: state['last_capacity_update_ts']=tnow
        pr = predict_probs_row(X_ref[i], Tcal)

        cands=[(ACTIONS[c], float(pr[c])) for c in range(len(ACTIONS))
               if ACTIONS[c] != 'NO_OP' and float(pr[c]) >= thresholds.get(ACTIONS[c],{}).get('tau',0.5)]
        cands.sort(key=lambda x:-x[1])
        selected=[]
        for n,p in cands:
            if len(selected) >= TOP_K: break
            if any(frozenset({n,m}) in MUTEX for m,_ in selected):
                continue
            selected.append((n,p))

        if not selected:
            continue

        for name,score in selected:
            tau=thresholds.get(name,{}).get('tau',0.5)
            if score < tau:
                logs['gate_blocks'].append({'idx':int(i),'action':name,'p':score,'tau':tau,'reason':'below_tau'})
                continue
            if name in ['REQUEST_CONSULT','REQUEST_BED']:
                if not capacity_fresh(tnow, state.get('last_capacity_update_ts', None), max_age_min=20):
                    logs['policy_blocks'].append({'idx':int(i),'action':name,'reason':'capacity_stale'})
                    continue
                key=(int(i), name, 'IM')
                if key in state['sent_packets']:
                    logs['policy_blocks'].append({'idx':int(i),'action':name,'reason':'duplicate'})
                    continue
                state['sent_packets'].add(key)
            logs['proposals'].append({'idx':int(i),'action':name,'p':score,'ts':tnow})
            if name in ['REQUEST_CONSULT','REQUEST_BED']:
                if np.random.rand()<0.7:
                    logs['approvals'].append({'idx':int(i),'action':name,'ts':tnow})
            else:
                logs['approvals'].append({'idx':int(i),'action':name,'ts':tnow})
    return logs

# Calibrated temperature from CAL
@torch.no_grad()
def logits_on(splitX): return model(to_t(splitX)).cpu().numpy()
val_logits = logits_on(Xv_all)
def softmaxT(z, T=1.0, axis=1):
    zT = z / max(T,1e-6); zT -= zT.max(axis=axis, keepdims=True); e=np.exp(zT); return e/e.sum(axis=axis, keepdims=True)
probs_val = softmaxT(val_logits, T=bestT)

thr_cost = json.loads((ARTS/'thresholds.json').read_text())
thr_ops  = json.loads((ARTS/'thresholds_ops.json').read_text())

logs_cost = replay(thr_cost, bestT, Xv_all, yv_all, tv_all)
logs_ops  = replay(thr_ops,  bestT, Xv_all, yv_all, tv_all)

from collections import Counter
def summarize(logs, label):
    props=len(logs['proposals']); appr=len(logs['approvals'])
    print(f"{label}: proposals={props} approvals={appr} approve_rate={appr/max(1,props):.2f} "
          f"gate_blocks={len(logs['gate_blocks'])} policy_blocks={len(logs['policy_blocks'])}")

summarize(logs_cost, "Replay (PR/Cost, VAL)")
print("Policy blocks (PR):", Counter([b['reason'] for b in logs_cost['policy_blocks']]))
summarize(logs_ops,  "Replay (OPS, VAL)")
print("Policy blocks (OPS):", Counter([b['reason'] for b in logs_ops['policy_blocks'])])


In [ ]:

# ---- KPIs (VAL-only) + PR curves + Confusion Matrix; save to ARTS ----
def pager_load(logs, ts_all):
    pages = [p for p in logs["proposals"] if p["action"] in ["REQUEST_CONSULT","REQUEST_BED"]]
    if not pages: return {"pages":0,"per_hour":0.0}
    hours = max(1.0, (ts_all.max() - ts_all.min())/60.0)
    return {"pages":len(pages), "per_hour": round(len(pages)/hours, 3)}

def simulate_capacity_updates_list(total_minutes):
    t=0; out=[]
    while t<total_minutes:
        out.append(t); t+=int(np.random.uniform(10,20))
    return out

def minutes_saved_for(action, ts_now, last_cap_update):
    hour = (ts_now//60)%24
    is_day = 8 <= hour < 20
    if action in ['ORDER_CT','PERFORM_FAST']:
        base = 15.0 if is_day else 30.0
    elif action in ['REQUEST_CONSULT','REQUEST_BED']:
        base = 4.0 if is_day else 6.0
    elif action in ['ORDER_ECG','ORDER_LABS']:
        base = 2.0 if is_day else 3.0
    elif action in ['ORDER_XR']:
        base = 1.5 if is_day else 2.5
    else:
        base = 0.0
    if last_cap_update is None:
        return 0.0
    delta = ts_now - last_cap_update
    if delta <= 0: discount = 1.0
    elif delta >= 20: discount = 0.0
    else: discount = 1.0 - (delta/20.0)
    return max(0.0, base * discount)

def kpis_from_logs(logs, ts_all):
    props = logs["proposals"]; appr = logs["approvals"]
    approve_rate = len(appr)/max(1,len(props))
    page_props = [p for p in props if p["action"] in ["REQUEST_CONSULT","REQUEST_BED"]]
    page_appr  = [a for a in appr  if a["action"] in ["REQUEST_CONSULT","REQUEST_BED"]]
    fpr = (len(page_props)-len(page_appr))/max(1,len(page_props))
    caps = simulate_capacity_updates_list(SIM_MINUTES)
    saved = []
    for p in props:
        last_update = max([c for c in caps if c <= p["ts"]], default=None)
        saved.append(minutes_saved_for(p["action"], p["ts"], last_update))
    med_saved = float(np.median(saved)) if saved else 0.0
    return {"proposals":len(props),"approvals":len(appr),"approve_rate":round(approve_rate,3),
            "false_page_rate":round(fpr,3), "median_minutes_saved":round(med_saved,2)}

kpi_report = {"PR_cost": kpis_from_logs(logs_cost, tv_all),
              "OPS":      kpis_from_logs(logs_ops,  tv_all)}
(ARTS/'kpi_report.json').write_text(json.dumps(kpi_report, indent=2))
print("KPI report:", kpi_report)
print("Pager load (PR):", pager_load(logs_cost, tv_all))
print("Pager load (OPS):", pager_load(logs_ops,  tv_all))

# Plots: guard PR curves if zero positives
from sklearn.metrics import precision_recall_curve
critical = ['ORDER_CT','PERFORM_FAST','ORDER_ECG']
plt.figure()
for name in critical:
    ci = ACTIONS.index(name)
    yb = (yv_all==ci).astype(int)
    if yb.sum() == 0:
        print(f"[PR] Skipping {name}: zero positives in VAL")
        continue
    prec, rec, thr = precision_recall_curve(yb, probs_val[:,ci])
    plt.plot(rec, prec, label=name)
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("PR Curves (critical actions)"); plt.legend()
plt.tight_layout(); plt.savefig(ARTS/'pr_curves_val.png', dpi=160); plt.close()

# Confusion matrix
y_pred = probs_val.argmax(axis=1)
cm = np.zeros((len(ACTIONS), len(ACTIONS)), dtype=int)
for t,p in zip(yv_all, y_pred): cm[t,p]+=1
plt.figure()
plt.imshow(cm, interpolation="nearest")
plt.title("Val Confusion Matrix"); plt.xlabel("Pred"); plt.ylabel("True")
plt.xticks(range(len(ACTIONS)), ACTIONS, rotation=45, ha="right"); plt.yticks(range(len(ACTIONS)), ACTIONS)
plt.colorbar()
plt.tight_layout(); plt.savefig(ARTS/'cm_val.png', dpi=160); plt.close()

print("Saved plots:", str(ARTS/'pr_curves_val.png'), str(ARTS/'cm_val.png'))


In [ ]:

# ---- Run summary + audit anchor, then list artifacts with sizes ----
summary = {
    "preset": PRESET,
    "target_daily_arrivals": TARGET_DAILY_ARRIVALS,
    "lambda_scale": float(LAMBDA_SCALE),
    "n_days": N_DAYS,
    "top_k": TOP_K,
    "use_focal": USE_FOCAL,
    "gamma": 2.0,
    "recall_floor": RECALL_FLOOR,
    "fp_budget_per_h": FP_BUDGET_PER_H,
}
(ARTS/'run_summary.json').write_text(json.dumps(summary, indent=2))

import hashlib, time, glob
def file_hash(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

asset_paths = sorted([str(p) for p in ARTS.glob("*.*")])
leaves = [file_hash(p) for p in asset_paths]
layer = leaves[:]
if not layer: root = hashlib.sha256(b"").hexdigest()
else:
    while len(layer)>1:
        nxt = []
        it = iter(layer)
        for a in it:
            b = next(it, a)
            nxt.append(hashlib.sha256((a+b).encode()).hexdigest())
        layer = nxt
    root = layer[0]

anchor = {"root":root,"ts":time.strftime("%Y-%m-%dT%H:%M:%SZ")}
with open(ARTS/"audit_anchor.log","a") as f: f.write(json.dumps(anchor)+"\n")
print("Anchored:", anchor)

# List artifacts
from pathlib import Path
def list_artifacts(arts_dir):
    rows=[]
    for p in sorted(Path(arts_dir).glob("*")):
        if p.is_file():
            rows.append((p.name, p.stat().st_size))
    return rows

files = list_artifacts(ARTS)
print("Artifacts:", files)
